In [ ]:
from datasets import load_dataset
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm
from langchain.docstore.document import Document
from tqdm import tqdm
import csv
import os
from langchain_cohere import CohereRerank
from flair.data import Sentence
from flair.models import SequenceTagger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

lang = 'fi'

model_name = "intfloat/multilingual-e5-small"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

vectordb = Chroma(persist_directory=f"e5_corpus_{lang}", embedding_function = hf)

print("Preparing data")
id_list = []
with open(f'e5_{lang}.txt', 'r') as f:
    for line in f:
        info = line.split()
        if int(info[3]) > 300: continue
        id_list.append(info[2])

doc_list = []
for id in tqdm(id_list):
    doc = vectordb.get(id)
    doc_list.append(Document(page_content=doc['documents'][0][9:], metadata = {'id': doc['ids'][0]}))


compressor = CohereRerank(model = 'rerank-multilingual-v3.0', top_n = 300)

tagger = SequenceTagger.load(f"hmbert/flair-hipe-2022-newseye-{lang}")


judge_list = []
with open(f"miracl-v1.0-{lang}_qrels_qrels.miracl-v1.0-{lang}-dev.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    for row in rd:
        judge_list.append([row[0], row[2], int(row[3])])

query_list = []
with open(f"miracl-v1.0-{lang}_topics_topics.miracl-v1.0-{lang}-dev.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    for row in rd:
        query_list.append(row)


def ner_extract(text):
    try:
        sentence = Sentence(text)
        tagger.predict(sentence)
        sen_dict = sentence.to_dict(tag_type='ner')
        ner = " ".join([ner['labels'][0]['value'] for ner in sen_dict['entities']] + ['O'])
    except:
        return "O"
    return ner

import pickle

retrieved_docs_list = []

f = open(f"ner_{lang}.txt", "a")

print("Evaluating")

for i, (query_id, query) in tqdm(enumerate(query_list)):
    vectorizer = TfidfVectorizer()
    query = f'query: {query}'

    if i > 0 and i % 10 == 0:
        time.sleep(60)

    temp = 300 * i
    docs = doc_list[temp:temp+300]
    docs = compressor.compress_documents(docs, query)
    ids = [doc.metadata['id'] for doc in docs]
    doc_contents = [doc.page_content for doc in docs]
    
    aner = ner_extract(query[7:])
    if aner != "O":
        ners = [ner_extract(doc.page_content) for doc in docs]
        all_ner = ners + [aner]
        tfidf_matrix = vectorizer.fit_transform(all_ner)
        query_vector = tfidf_matrix[-1]
        doc_vectors = tfidf_matrix[:-1]
        ner_scores = cosine_similarity(query_vector, doc_vectors).flatten()
        co_scores = np.array([float(doc.metadata['relevance_score']) for doc in docs])

        scores = 0.8 * co_scores + 0.2 * ner_scores
        max_idx = np.argsort(-scores)
    else:
        co_scores = np.array([float(doc.metadata['relevance_score']) for doc in docs])
        scores = co_scores
        max_idx = np.argsort(-scores)
    
    top_docs = []
    final_docs = []
    for idx in max_idx[:100]:
        final_docs.append((ids[idx], scores[idx]))
        top_docs.append(doc_contents[idx])
    
    retrieved_docs_list.append(top_docs)

    for j, (doc, score) in enumerate(final_docs):
        f.write(f'{query_id} Q0 {doc} {j + 1} {score} ner\n')

f.close()

with open(f'retrieve_{lang}', 'wb') as fp:
    pickle.dump(retrieved_docs_list, fp)


In [ ]:
import pytrec_eval
from collections import defaultdict
import json

results = defaultdict(dict)
with open(f"ner_{lang}.txt", 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 6:
            query_id = parts[0]
            doc_id = parts[2]
            score = float(parts[4])
            results[query_id][doc_id] = score

qrels = defaultdict(dict)
with open(f"miracl-v1.0-{lang}_qrels_qrels.miracl-v1.0-{lang}-dev.tsv", 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    for row in reader:
        if len(row) >= 4:
            query_id = row[0]
            doc_id = row[2]
            relevance = int(row[3])
            qrels[query_id][doc_id] = relevance

metrics = {'ndcg_cut_10', 'recall_100'}
evaluator = pytrec_eval.RelevanceEvaluator(dict(qrels), metrics)
scores = evaluator.evaluate(dict(results))

ndcg_10_scores = [s.get('ndcg_cut_10', 0.0) for s in scores.values()]
recall_100_scores = [s.get('recall_100', 0.0) for s in scores.values()]

avg_ndcg_10 = sum(ndcg_10_scores) / len(ndcg_10_scores)
avg_recall_100 = sum(recall_100_scores) / len(recall_100_scores)

print(f"NDCG@10: {avg_ndcg_10:.4f}")
print(f"Recall@100: {avg_recall_100:.4f}")
